# M29 — Understand Attention Through Context

**Objective:** experience attention as context-dependent relevance weighting.

A vector from M28 is frozen. The useful whole here is a **tiny single-head
attention pass** over three tokens:

`X → Q, K, V → scaled dots → mask → softmax over keys → weighted values`

The same middle token `bank` sits in `river bank cash` and
`river bank water`. Only the third representation changes. If attention is
doing its job, the **output at `bank`** should move.

Multi-head split/merge, residuals, normalization, and the feed-forward
sublayer stay closed (M30). Nothing is downloaded.


## Working contract

Every experiment follows **predict → act → observe → explain**. **Predict before running**
each action cell and timestamp the prediction in your own evidence log.
A prediction is falsifiable: a key id, a weight order, a row sum, a zero
on a masked position, or a changed output coordinate.

Attention weights are not a causal explanation of intent. Do not
open a transformer block. Softmax is an axis choice, not a vibe.

The repository does not prefill learner answers, ADR text, or competence.


In [ ]:
from pathlib import Path
import inspect
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "missions" / "M29" / "attention_core.py").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Run from the LearningOS-AI repository or its labs directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from missions.M29.attention_core import (
    BANK_INDEX,
    CASH_TOKENS,
    CONTEXT_INDEX,
    HAND_K,
    HAND_Q,
    HAND_V,
    IDENTITY_2,
    INTERPRETATION_LIMIT,
    TEACHING_W_K,
    TEACHING_W_Q,
    TEACHING_W_V,
    WATER_TOKENS,
    X_CASH_CONTEXT,
    X_WATER_CONTEXT,
    aggregate_values,
    attention_with_defect,
    causal_additive_mask,
    dot_product_scores,
    observability_report,
    project_qkv,
    repair_attention,
    replace_position,
    scale_fixture_qkv,
    scale_scores,
    scaled_dot_product_attention,
    self_attention,
    softmax_over_keys,
    teaching_batch,
    weight_invariants,
)

print("repository root:", ROOT)
print("cash tokens:", CASH_TOKENS)
print("water tokens:", WATER_TOKENS)
print("d_model:", np.asarray(X_CASH_CONTEXT).shape[-1])
print("scale default: 1/sqrt(d_k)")
print("interpretation limit:", INTERPRETATION_LIMIT)


## M16 / M28 boundary: matrices and static vectors in, the block out

M16 already maps a row-batch with `X @ W` and treats a transpose as a
convention repair. M28 already ranks **frozen** sentence vectors. Neither
mission lets a position's representation depend on its neighbors.

What this mission **opens:** queries, keys, values, scaled dot-product
scores, masks, softmax over keys, weighted value aggregation, and
context-dependent outputs.

What stays **deferred:**
- M30 — multi-head split/merge, residual, LayerNorm placement, FFN
- M31 — LLM training
- M33 — a retrieval service

A cosine from M28 is not an attention weight. Do not relabel it.


## Frozen teaching fixtures

Declare the useful whole **before** the first score.

| Fixture | Value |
| --- | --- |
| Sequences | `river bank cash` vs `river bank water` |
| `river` | `(2.0, 0.0)` nature axis |
| `bank` | `(1.0, 1.0)` ambiguous |
| `cash` | `(0.0, 2.0)` finance axis |
| `water` | `(3.0, 0.0)` stronger nature |
| Projections (first pass) | identity, so `Q = K = V = X` |
| Scale | `1/sqrt(d_k)` with `d_k = 2` |
| Softmax axis | keys (last axis) |
| Hand micro-case | `q=(1,0)`, `k0=(1,0)`, `k1=(0,1)`, `v0=(1,0)`, `v1=(0,2)` |
| Download | false — numbers live in `attention_core.py` |

Primary sources: `hf-llm-course` and `karpathy-zero-to-hero` in
`data/source_registry.json`. Skip pretrained transformers here.

The table is authored so context, masks, and axis bugs are visible. It
is not a language-model quality benchmark.


In [ ]:
cash_x = np.asarray(X_CASH_CONTEXT, dtype=float)
water_x = np.asarray(X_WATER_CONTEXT, dtype=float)
batch_x = teaching_batch()
print("cash X shape", cash_x.shape)
print("cash X\n", cash_x)
print("water X\n", water_x)
print("named change is index", CONTEXT_INDEX, "only")
print("batch shape", batch_x.shape, "(cash row 0, water row 1)")
print("bank vector", cash_x[BANK_INDEX], "equals water bank", water_x[BANK_INDEX])


### Two sequences, one named change

Row-batch layout is M16: one token per row, two features. The bank row
is identical in both sequences. If a static embedding table were enough,
bank's output would be the same after mixing neighbors. Attention is the
claim that it need not be.

Keep the vectors. Predict the whole pass before any softmax.


## Predict before running — useful whole

Timestamp a prediction before `run-whole`.

Identity projections, scale `1/sqrt(d_k)`, no mask. The only change
between the two sequences is the third token vector.

Predict:
- whether bank's **output vector** stays the same in cash vs water
- which key should receive the most mass from the bank query in the
  cash sequence
- which key should receive the most mass from the bank query in the
  water sequence

Do not compute dots yet. A geometric guess from the table is the point.


In [ ]:
cash = self_attention(X_CASH_CONTEXT, tokens=CASH_TOKENS)
water = self_attention(X_WATER_CONTEXT, tokens=WATER_TOKENS)
batch = self_attention(teaching_batch())
print("cash shapes", cash.shapes)
print("cash bank weights", cash.weights[0, BANK_INDEX])
print("water bank weights", water.weights[0, BANK_INDEX])
print("cash bank output", cash.output[0, BANK_INDEX])
print("water bank output", water.output[0, BANK_INDEX])
print("batch bank outputs\n", batch.output[:, BANK_INDEX])
print("cash report", observability_report(cash))
assert np.allclose(cash.weights[0, BANK_INDEX], 1.0 / 3.0)
assert water.weights[0, BANK_INDEX, CONTEXT_INDEX] > cash.weights[0, BANK_INDEX, CONTEXT_INDEX]
assert not np.allclose(cash.output[0, BANK_INDEX], water.output[0, BANK_INDEX])
assert np.allclose(batch.output[0], cash.output[0])
assert np.allclose(batch.output[1], water.output[0])
print("context changed the bank representation; weights are not intent")


### Context changed the representation

In the cash sequence the bank query scores all three keys equally, so
the weights are uniform and the output stays `(1, 1)`. Replacing cash
with water raises one key score. Bank's output moves toward the nature
axis. That is the useful whole: **same token identity, different
context vector**.

The numbers did not "understand banking." They mixed value vectors
using softmax-normalized dots. Next, name Q, K, and V as matrices.


## Predict before running — Q, K, V shapes

Timestamp a prediction before `run-qkv`.

`X` is `(3, 2)`. Identity `W_Q`, `W_K`, `W_V` are `(2, 2)`. The core
adds a batch axis, so projections are `(batch, seq, dim)`.

Predict:
- shapes of Q, K, and V after identity projection
- whether identity Q equals `X`
- whether the teaching matrices `TEACHING_W_Q` / `TEACHING_W_K`
  leave Q and K unchanged

This is M16 row-batch multiplication, not a metaphor.


In [ ]:
q, k, v = project_qkv(X_CASH_CONTEXT, IDENTITY_2, IDENTITY_2, IDENTITY_2)
print("identity Q shape", q.shape, "K", k.shape, "V", v.shape)
print("identity Q equals X", np.allclose(q[0], cash_x))
qt, kt, vt = project_qkv(X_CASH_CONTEXT, TEACHING_W_Q, TEACHING_W_K, TEACHING_W_V)
print("taught W_Q\n", np.asarray(TEACHING_W_Q))
print("taught Q\n", qt[0])
print("taught K\n", kt[0])
print("taught V equals identity V", np.allclose(vt, v))
print("Q changed", not np.allclose(qt, q), "K changed", not np.allclose(kt, k))
taught = scaled_dot_product_attention(qt, kt, vt, tokens=CASH_TOKENS)
print("taught bank weights", taught.weights[0, BANK_INDEX])
assert q.shape == (1, 3, 2)
assert not np.allclose(qt, q)


### Three views of the same sequence

Identity projections make Q, K, and V copies of `X`, which is why the
useful whole could be read as self-attention on the token table. The
teaching matrices change Q and K without touching V. Scores can move
while values stay put — that split is the next experiment's backbone.


## Predict before running — raw and scaled dots

Timestamp a prediction before `run-scores`.

Hand micro-case, unscaled: `q=(1,0)` against `k0=(1,0)` and `k1=(0,1)`.
Cash-context bank against river, bank, cash. Water-context bank against
river, bank, water.

Predict:
- the two raw hand scores, and whether the first softmax weight exceeds 0.5
- whether cash-context bank has a unique argmax
- which water-context key has the highest raw score
- whether dividing by `1/sqrt(d_k)` can change that argmax

Do not invoke a transformer. This is dots and a scale.


In [ ]:
hand = scaled_dot_product_attention(HAND_Q, HAND_K, HAND_V, scale="none")
print("hand raw scores", hand.raw_scores[0, 0])
print("hand weights", hand.weights[0, 0])
print("hand output", hand.output[0, 0])
print("hand row sum", float(hand.weights[0, 0].sum()))

cash_raw = dot_product_scores(cash_x, cash_x)
water_raw = dot_product_scores(water_x, water_x)
cash_scaled = scale_scores(cash_raw, d_k=2, scale="dk")
water_scaled = scale_scores(water_raw, d_k=2, scale="dk")
print("cash raw\n", cash_raw[0])
print("water raw\n", water_raw[0])
print("cash scaled bank", cash_scaled[0, BANK_INDEX])
print("water scaled bank", water_scaled[0, BANK_INDEX])
print("scale multiplier", 1.0 / np.sqrt(2.0))

water_weights = softmax_over_keys(water_scaled)
water_from_parts = aggregate_values(water_weights, water_x)
print("softmax over keys, bank", water_weights[0, BANK_INDEX])
print("aggregated bank", water_from_parts[0, BANK_INDEX])
assert np.allclose(water_from_parts, water.output)
assert np.argmax(water_raw[0, BANK_INDEX]) == CONTEXT_INDEX
assert np.argmax(water_scaled[0, BANK_INDEX]) == CONTEXT_INDEX


### Scores, then weights, then output

The hand case `softmax([1, 0])` puts more than half the mass on key 0
because `e > 1`. Scaling by `1/sqrt(d_k)` flattens that a little but
does not change the argmax. Cash-context bank is a three-way tie.
Water-context bank is not. Softmax over keys turns those scores into a
row that sums to 1; `weights @ V` is the output.

If you ever softmax the other axis, that row-sum contract dies. Hold
that thought for the controlled failure.


## Predict before running — causal mask

Timestamp a prediction before `run-causal`.

Same cash sequence, same Q/K/V/scale. The named change is a causal
mask: position `i` may not put mass on key `j > i`.

Predict:
- bank's weight on `cash` (the future token)
- whether bank's remaining two weights still sum to 1
- whether the pre-mask scores stay equal to the unmasked cash scores
- whether position 0 can see `bank`

Forbidden mass should be diagnosed from the mask, not from a story
about the model "refusing to look ahead."


In [ ]:
mask = causal_additive_mask(3)
causal = self_attention(X_CASH_CONTEXT, mask=mask, tokens=CASH_TOKENS)
print("causal weights\n", causal.weights[0])
print("causal bank output", causal.output[0, BANK_INDEX])
print("invariants", weight_invariants(causal.weights, mask=mask))
print("pre-mask scores equal", np.allclose(causal.scaled_scores, cash.scaled_scores))
print("unmasked bank cash mass", cash.weights[0, BANK_INDEX, CONTEXT_INDEX])
assert np.allclose(causal.weights[0, 0], (1.0, 0.0, 0.0))
assert np.allclose(causal.weights[0, BANK_INDEX], (0.5, 0.5, 0.0))
assert np.allclose(causal.output[0, BANK_INDEX], (1.5, 0.5))
assert causal.invariants()["future_mass_zero"]


### Future keys carry no mass; allowed keys renormalize

Position 0 can only see itself. Bank can see river and itself, not
cash, so a former three-way tie becomes `1/2, 1/2, 0` and the output
is `0.5 * river + 0.5 * bank = (1.5, 0.5)`. The mask was added
**before** softmax. Zeroing after softmax is a different bug.


## Predict before running — scale factor

Timestamp a prediction before `run-scale`.

A separate fixture uses `d_k = 8`. Query is all ones. Key 0 is all
ones (raw score 8). Key 1 flips the second half of the coordinates
(raw score 0). Q and K stay fixed. The named change is the scale:
unscaled (`scale="none"`) versus `1/sqrt(d_k)`.

Predict:
- which setting puts more mass on key 0
- whether the argmax changes
- whether entropy of the two-key distribution is higher after scaling

Large unscaled dots saturate softmax. That is why Vaswani-style scale
exists; it is not mysticism.


In [ ]:
q8, k8, v8 = scale_fixture_qkv()
unscaled = scaled_dot_product_attention(q8, k8, v8, scale="none")
scaled8 = scaled_dot_product_attention(q8, k8, v8, scale="dk")
print("raw scores", unscaled.raw_scores[0, 0])
print("unscaled weights", unscaled.weights[0, 0], "entropy", unscaled.invariants()["entropy"])
print("scaled weights", scaled8.weights[0, 0], "entropy", scaled8.invariants()["entropy"])
print("unscaled scale", unscaled.scale, "dk scale", scaled8.scale)
assert unscaled.weights[0, 0, 0] > scaled8.weights[0, 0, 0]
assert float(unscaled.invariants()["entropy"].reshape(-1)[0]) < float(scaled8.invariants()["entropy"].reshape(-1)[0])
assert np.allclose(unscaled.q, scaled8.q) and np.allclose(unscaled.k, scaled8.k)


### Scale changes concentration, not the story

Both settings still prefer the aligned key. Unscaled `softmax([8, 0])`
is almost one-hot. Dividing by `sqrt(8)` leaves a visible tail. If a
later block (M30) changes head width, this scale travels with `d_k`.


## Predict before running — query perturbation

Timestamp a prediction before `run-qk-perturb`.

Cash-context K and V stay fixed. The named change is the bank **query**:
replace `(1, 1)` with `(1.5, 0.5)` (a step toward river).

Predict:
- the three raw scores for the moved bank query
- whether river's weight rises
- whether cash's weight falls
- whether V changed

If scores move and values do not, the mix changes because the query
pointed somewhere else, not because a value vector was edited.


In [ ]:
q_moved = replace_position(cash.q, BANK_INDEX, (1.5, 0.5))
moved = scaled_dot_product_attention(q_moved, cash.k, cash.v)
print("base bank scores", cash.raw_scores[0, BANK_INDEX])
print("moved bank scores", moved.raw_scores[0, BANK_INDEX])
print("base bank weights", cash.weights[0, BANK_INDEX])
print("moved bank weights", moved.weights[0, BANK_INDEX])
print("V unchanged", np.allclose(moved.v, cash.v))
assert np.allclose(moved.raw_scores[0, BANK_INDEX], (3.0, 2.0, 1.0))
assert moved.weights[0, BANK_INDEX, 0] > cash.weights[0, BANK_INDEX, 0]
assert moved.weights[0, BANK_INDEX, 2] < cash.weights[0, BANK_INDEX, 2]


### Queries steer scores; values wait

The moved query scores river 3, itself 2, cash 1. Mass follows the
scores. Values never changed, so any output movement is a different
weighting of the same table. The complementary experiment edits values
only.


## Predict before running — value-only change

Timestamp a prediction before `run-value`.

Cash-context Q, K, mask, and scale stay fixed. Replace only the cash
**value** with `(0, 4)`.

Predict:
- whether bank weights change
- whether bank output changes
- the new bank output if the weights really stay `1/3` each

This is the discriminator between "attention looked somewhere else"
and "the thing it mixed was edited."


In [ ]:
new_v = replace_position(cash.v, CONTEXT_INDEX, (0.0, 4.0))
value_changed = scaled_dot_product_attention(cash.q, cash.k, new_v)
print("weights equal", np.allclose(value_changed.weights, cash.weights))
print("scores equal", np.allclose(value_changed.raw_scores, cash.raw_scores))
print("base bank output", cash.output[0, BANK_INDEX])
print("new bank output", value_changed.output[0, BANK_INDEX])
print("expected if 1/3 mix", (1.0, 5.0 / 3.0))
assert np.allclose(value_changed.weights, cash.weights)
assert np.allclose(value_changed.output[0, BANK_INDEX], (1.0, 5.0 / 3.0))


### Weights can stay fixed while the output moves

Scores live in Q/K space. The output lives in V space. Editing a value
is allowed to change the mix's result without changing the mix's
coefficients. If a debugging session treats a changed output as proof
that "attention moved," this experiment falsifies that leap.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2))
panels = (
    (cash.weights[0], "cash context, no mask"),
    (water.weights[0], "water context, no mask"),
    (causal.weights[0], "cash context, causal"),
)
for axis, (matrix, title) in zip(axes, panels):
    image = axis.imshow(matrix, vmin=0.0, vmax=1.0, cmap="viridis")
    axis.set_title(title)
    axis.set_xlabel("key")
    axis.set_ylabel("query")
    axis.set_xticks(range(3), list(CASH_TOKENS if "water" not in title else WATER_TOKENS))
    axis.set_yticks(range(3), list(CASH_TOKENS if "water" not in title else WATER_TOKENS))
    fig.colorbar(image, ax=axis, fraction=0.046)
fig.suptitle("Attention weights (softmax over keys)")
fig.tight_layout()
plt.show()
plt.close(fig)
print("max cash", float(cash.weights.max()), "max water", float(water.weights.max()))
print("causal future mass", float(causal.weights[0, 0, 1] + causal.weights[0, 0, 2] + causal.weights[0, 1, 2]))


## Code reading — project, score, mask, softmax, aggregate

Read `project_qkv`, `dot_product_scores`, `scale_scores`,
`apply_additive_mask`, `softmax_over_keys`, `aggregate_values`, and
`scaled_dot_product_attention` in `missions/M29/attention_core.py`.

Trace, then predict before the next cell:

1. shapes of Q, K, V, scores, weights, and output for cash `(3, 2)`
   with identity projections
2. whether softmax runs over keys or over queries
3. whether a causal mask at position 0 can put mass on `cash`

Do not search the file for head splitting, a residual add, or a
feed-forward width. Stay on one head and one axis.


In [ ]:
attn_src = inspect.getsource(scaled_dot_product_attention)
soft_src = inspect.getsource(softmax_over_keys)
proj_src = inspect.getsource(project_qkv)
print("projection uses matmul", "@" in proj_src)
print("softmax_over_keys axis marker", "KEY_AXIS" in soft_src or "axis=KEY_AXIS" in soft_src or "axis=-1" in soft_src)
print("attention calls softmax_over_keys", "softmax_over_keys" in attn_src)
print("mask applied in attention", "apply_additive_mask" in attn_src)
print("repair_attention recomputes from the trace", "trace.q" in inspect.getsource(repair_attention))
print("cash shapes", cash.shapes)
print("softmax over keys on a (1, 3, 3) matrix reduces the last axis")
print("position 0 causal row", causal.weights[0, 0])
assert cash.shapes["q"] == (1, 3, 2)
assert cash.shapes["scores"] == (1, 3, 3)
assert cash.shapes["output"] == (1, 3, 2)
assert causal.weights[0, 0, CONTEXT_INDEX] == 0.0


## Predict before running — Controlled failure: softmax axis

Timestamp a prediction before `run-failure`.

Cash-context Q, K, V, and scale stay fixed. The defective path sets
`defect="softmax_over_queries"` (softmax over the query axis). A second
named defect, `defect="mask_after_softmax"`, keeps causal keep-masking
but applies it after softmax without renormalizing.

Predict:
- whether each **query row** of the wrong-axis weights still sums to 1
- whether each **key column** might sum to 1 instead
- whether mask-after-softmax can zero the future and still have rows
  that sum to 1
- whether the two-key hand case would still match `e/(e+1)` on the
  broken axis

The tensors can look like probabilities. That is the trap.


In [ ]:
broken_axis = attention_with_defect(
    cash.q, cash.k, cash.v, defect="softmax_over_queries"
)
print("axis defect", broken_axis.defect, "softmax_axis", broken_axis.softmax_axis)
print("wrong-axis weights\n", broken_axis.weights[0])
print("row sums", broken_axis.weights.sum(axis=-1))
print("col sums", broken_axis.weights.sum(axis=-2))
print("wrong-axis invariants", broken_axis.invariants())
print("scores still equal", np.allclose(broken_axis.scaled_scores, cash.scaled_scores))

broken_mask = attention_with_defect(
    cash.q, cash.k, cash.v, mask=mask, defect="mask_after_softmax"
)
print("mask defect timing", broken_mask.mask_timing)
print("mask-after weights\n", broken_mask.weights[0])
print("mask-after row sums", broken_mask.weights.sum(axis=-1))
print("mask-after invariants", weight_invariants(broken_mask.weights, mask=mask))
assert broken_axis.invariants()["rows_sum_to_one"] is False
assert np.allclose(broken_axis.weights.sum(axis=-2), 1.0)
assert abs(float(broken_mask.weights[0, BANK_INDEX].sum()) - 1.0) > 1e-6


### Diagnose before repair

Symptom: finite nonnegative weights appeared. One table has columns
that sum to 1 and rows that do not. Another table zeros the future but
leaves bank's row summing to `2/3`.

Hypotheses worth separating: the token table changed; scale changed;
softmax reduced queries instead of keys; the mask was applied after
softmax and never renormalized.

The discriminating observations are the row-sum invariant, the column
sums, and the pre-mask scores matching the healthy cash trace. The
root cause is an axis or an order, not a missing transformer block.

Do not repair this by adding heads or a residual.


## Predict before running — smallest repair

Timestamp a prediction before `run-failure-repair`.

The repair must start from the **broken trace** (`repair_attention`),
not from a second unrelated happy-path call. Q, K, V, mask, and scale
stay the broken tensors.

Predict:
- whether `softmax_over_keys(broken.masked_scores)` matches the repair
- whether row sums return to 1
- whether a freshly broken call still fails the invariant (regression)

Smallest repair: softmax over keys, mask before softmax.


In [ ]:
from_broken = softmax_over_keys(broken_axis.masked_scores)
repaired = repair_attention(broken_axis)
print("repair matches key-softmax of broken scores", np.allclose(repaired.weights, from_broken))
print("repair matches original cash", np.allclose(repaired.weights, cash.weights))
print("repaired invariants", repaired.invariants())
print("Q/K/V unchanged from broken", np.allclose(repaired.q, broken_axis.q))

repaired_mask = repair_attention(broken_mask)
print("repaired causal weights\n", repaired_mask.weights[0])
print("repaired causal invariants", repaired_mask.invariants())

still_broken = attention_with_defect(
    broken_axis.q, broken_axis.k, broken_axis.v, defect="softmax_over_queries"
)
print("regression still broken", still_broken.invariants()["rows_sum_to_one"])
assert np.allclose(repaired.weights, from_broken)
assert repaired.invariants()["rows_sum_to_one"]
assert repaired_mask.invariants()["future_mass_zero"]
assert still_broken.invariants()["rows_sum_to_one"] is False
print("softmax over keys restored from the broken path")


### Restore the axis and the order; leave the tensors

The healthy weights were sitting in the broken trace's scores the whole
time. Repair renormalized over keys. A second call with the named
defect still fails, which is the regression. M30 can consume this
single-head path; it does not get to redefine softmax.


## Evidence contract

Submit, in your own log (not in this repository):

- timestamped **Predict before running** notes
- cash vs water bank weights and outputs
- Q/K/V shapes for identity and teaching projections
- hand micro-case vs `softmax([1, 0])`
- causal future-mass zero and renormalized bank row
- unscaled vs `1/sqrt(d_k)` concentration
- query perturbation and value-only split
- wrong-axis / mask-order diagnosis and `repair_attention`

See `missions/M29/evidence_contract.yaml`. Do not paste filled evidence
into the committed notebook.


## No-AI gate

Close this notebook and complete `missions/M29/no_ai_gate.md` from a blank
page without AI-generated code, calculations, prose, or diagrams.

Use only the fresh `q, K, V` in that file. Compute one query's
distribution and output, apply a causal mask by hand, diagnose a
wrong-axis result from invariants, and state what attention weights do
not prove.

**Status:** [UNFILLED BY LEARNER]


## Unfilled ADR

Use `missions/M29/adr_prompt.md` to choose a V06 attention-trace
observability policy (checkpoints, shapes, mask logging, invariants,
over-interpretation limits). Do not claim a production profiler.

- **Status:** [UNFILLED BY LEARNER]
- **Date:** [UNFILLED BY LEARNER]
- **Owner:** [UNFILLED BY LEARNER]
- **Decision:** [UNFILLED BY LEARNER]

This notebook is not that ADR. Formal engineering review is required
at M29 for the package, not as a substitute for the learner ADR.


## M16 → M28 → M29 handoff

M16 supplied row-batch matrix maps. M28 supplied static vectors and
the warning that a score is not meaning. M29 turns a sequence into Q,
K, and V and mixes values with softmax over keys.

M30 may compose this single head into a block. It must not relabel
these weights as multi-head semantics, a residual, or a norm
convention.

M31 and M33 stay closed. There is no training loop and no search
service here.

Reusable artifacts: `AttentionTrace` checkpoints
(`q`, `k`, `v`, `raw_scores`, `scaled_scores`, `masked_scores`,
`weights`, `output`), causal/padding additive masks, and the
cash/water teaching sequences.


## Mission summary prompt

In your own words, using only observations from this lab:

1. Why is `bank` a different vector after `cash` becomes `water`?
2. What invariant tells you softmax ran over keys?
3. Why can a value-only edit change the output without moving weights?
4. What must M30 receive that an output tensor without Q/K/V/mask
   checkpoints cannot provide?

Leave the answers in your evidence log, not in this file.


In [ ]:
assert np.allclose(cash.weights[0, BANK_INDEX], 1.0 / 3.0)
assert water.weights[0, BANK_INDEX, CONTEXT_INDEX] > cash.weights[0, BANK_INDEX, CONTEXT_INDEX]
assert np.allclose(hand.raw_scores[0, 0], (1.0, 0.0))
assert hand.weights[0, 0, 0] > 0.5
assert np.allclose(causal.weights[0, BANK_INDEX], (0.5, 0.5, 0.0))
assert np.allclose(causal.output[0, BANK_INDEX], (1.5, 0.5))
assert unscaled.weights[0, 0, 0] > scaled8.weights[0, 0, 0]
assert np.allclose(moved.raw_scores[0, BANK_INDEX], (3.0, 2.0, 1.0))
assert np.allclose(value_changed.weights, cash.weights)
assert np.allclose(value_changed.output[0, BANK_INDEX], (1.0, 5.0 / 3.0))
assert broken_axis.invariants()["rows_sum_to_one"] is False
assert np.allclose(repaired.weights, from_broken)
assert still_broken.invariants()["rows_sum_to_one"] is False
assert "not a causal explanation of intent" in INTERPRETATION_LIMIT
print("M29 integrity checks passed")
